In [1]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
import h5py

## Read and see the Dust dataset

In [6]:
import xarray as xr
import pandas as pd

fn = "/data/shared_data/DustProf_Proestakis/2007/Fine-Mode_Coarse-Mode_Pure-Dust-V1-CAL_LID_L2_05km-V4-21.2007-04-14T03-56-59ZN.nc"

groups = [
    "Geolocation",
    "Flags_and_Auxiliary",
    "EO4AQ-DustFM_Product/Backscatter_Coefficient_532",
    "EO4AQ-DustFM_Product/Extinction_Coefficient_532",
    "EO4AQ-DustFM_Product/Mass_Concentration",
]

ds_final = xr.merge([xr.open_dataset(fn, group=g) for g in groups])

raw_time = ds_final.Profile_UTC_Time.values
date_int = (raw_time // 1).astype(int) + 20000000
dt_series = (
    pd.to_datetime(date_int.astype(str), format="%Y%m%d")
    + pd.to_timedelta(raw_time % 1, unit="D")
)

ds_ = ds_final.assign_coords(
    time=("CAL_L2_5km_Pro", dt_series),
    lat=("CAL_L2_5km_Pro", ds_final.Latitude.values),
    lon=("CAL_L2_5km_Pro", ds_final.Longitude.values),
    height=("Alt", ds_final.Height.values),
).drop_vars(
    ["Profile_UTC_Time", "Latitude", "Longitude", "Height"]
)

ds_


<xarray.Dataset> Size: 40MB
Dimensions:                                       (CAL_L2_5km_Pro: 3148,
                                                   Alt: 399)
Coordinates:
    time                                          (CAL_L2_5km_Pro) datetime64[ns] 25kB ...
    lat                                           (CAL_L2_5km_Pro) float32 13kB ...
    lon                                           (CAL_L2_5km_Pro) float32 13kB ...
    height                                        (Alt) float64 3kB 29.84 ......
Dimensions without coordinates: CAL_L2_5km_Pro, Alt
Data variables:
    Day_Night_Flag                                (CAL_L2_5km_Pro) float32 13kB ...
    AVD_Aerosol_Subtype                           (CAL_L2_5km_Pro, Alt) float32 5MB ...
    AVD_Feature_Type                              (CAL_L2_5km_Pro, Alt) float32 5MB ...
    Surface_Elevation                             (CAL_L2_5km_Pro) float32 13kB ...
    Pure_Dust_Fine_Backscatter_Coefficient_532    (CAL_L2_5km_Pro, Alt) float32 5MB ...
    Pure_Dust_Coarse_Backscatter_Coefficient_532  (CAL_L2_5km_Pro, Alt) float32 5MB ...
    Pure_Dust_Fine_Extinction_Coefficient_532     (CAL_L2_5km_Pro, Alt) float32 5MB ...
    Pure_Dust_Coarse_Extinction_Coefficient_532   (CAL_L2_5km_Pro, Alt) float32 5MB ...
    Pure_Dust_Fine_Mass_Concentration             (CAL_L2_5km_Pro, Alt) float32 5MB ...
    Pure_Dust_Coarse_Mass_Concentration           (CAL_L2_5km_Pro, Alt) float32 5MB ...

## Pick the data over ENA and SGP region

In [2]:
import glob, os, numpy as np, xarray as xr, pandas as pd

base_in = "/data/shared_data/DustProf_Proestakis"
out_ena = "/data/ggong/CALIPSO/Dustprof/ENA_5x5"
out_sgp = "/data/ggong/CALIPSO/Dustprof/SGP_5x5"
os.makedirs(out_ena, exist_ok=True); os.makedirs(out_sgp, exist_ok=True)

groups = [
    "Geolocation",
    "Flags_and_Auxiliary",
    "EO4AQ-DustFM_Product/Backscatter_Coefficient_532",
    "EO4AQ-DustFM_Product/Extinction_Coefficient_532",
    "EO4AQ-DustFM_Product/Mass_Concentration",
]

ENA = (36.5916, 41.5916, -30.5257, -25.5257)
SGP = (34.107322, 39.107322, -99.987643, -94.987643)
def build_pretty(fn):
    dss = [xr.open_dataset(fn, group=g) for g in groups]
    ds = xr.merge(dss)
    for d in dss: d.close()
    rt = ds.Profile_UTC_Time.values
    di = (rt // 1).astype(int) + 20000000
    dt = pd.to_datetime(di.astype(str), format="%Y%m%d") + pd.to_timedelta(rt % 1, unit="D")
    return ds.assign_coords(
        time=("CAL_L2_5km_Pro", dt),
        lat=("CAL_L2_5km_Pro", ds.Latitude.values),
        lon=("CAL_L2_5km_Pro", ds.Longitude.values),
        height=("Alt", ds.Height.values),
    ).drop_vars(["Profile_UTC_Time", "Latitude", "Longitude", "Height"])

def save_box(ds, box, outdir, stem):
    latmin, latmax, lonmin, lonmax = box
    idx = np.where(
        (ds.lat >= latmin) & (ds.lat <= latmax) &
        (ds.lon >= lonmin) & (ds.lon <= lonmax)
    )[0]
    if idx.size:
        ds.isel(CAL_L2_5km_Pro=idx).to_netcdf(os.path.join(outdir, stem + ".nc"))

files = sorted(sum([glob.glob(f"{base_in}/{y}/*.nc") for y in range(2006, 2022)], []))
bad = []

for fn in files:
    try:
        stem = os.path.basename(fn).replace(".nc", "")
        ds = build_pretty(fn)
        save_box(ds, ENA, out_ena, stem + ".ENA")
        save_box(ds, SGP, out_sgp, stem + ".SGP")
        ds.close()
    except Exception:
        bad.append(fn)
        print("SKIP:", fn)

print("Done. Bad files:", len(bad))


SKIP: /data/shared_data/DustProf_Proestakis/2011/Fine-Mode_Coarse-Mode_Pure-Dust-V1-CAL_LID_L2_05km-V4-21.2011-08-29T10-44-18ZN.nc
SKIP: /data/shared_data/DustProf_Proestakis/2021/Fine-Mode_Coarse-Mode_Pure-Dust-V1-CAL_LID_L2_05km-V4-21.2021-03-09T22-04-51ZN.nc
Done. Bad files: 2
